# nb_gold_build

Builds the Gold star schema in lh_health_analytics from the Silver Delta tables.

Star: fact_measure_value + dim_hospital + dim_measure + dim_period + dim_reported_measure.

Load order is Kimball-standard: dimensions first, fact last (fact resolves each
dimension surrogate key by joining Silver on the business key). Surrogate keys are
generated in Spark with row_number() over an ordered window (deterministic 1..N).

Grain of fact_measure_value is one row per AIHW data item, so non-hospital
reporting units (peer-group / state / national / LHN aggregates) are retained for
benchmark context; those rows carry hospital_key = -1 (Unknown member).

See GOLD_NOTEBOOK.md for the walkthrough.

In [ ]:
# Cell 1 - Imports, gold schema, Silver schema discovery
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Silver is the schema source of truth - print actual columns before building
for t in ["datasets", "measures", "measure_categories", "reporting_units", "data_items"]:
    print(f"===== silver.{t} =====")
    spark.table(f"silver.{t}").printSchema()

# Eyeball a few rows of the four tables whose columns are not yet enumerated in the repo
for t in ["datasets", "measures", "measure_categories", "reporting_units"]:
    print(f"===== silver.{t} sample =====")
    spark.table(f"silver.{t}").show(3, truncate=False)

# Reporting-unit type spread in the facts (confirms grain decision M3-17 risk 2)
print("===== data_items reporting_unit_type_code spread =====")
spark.table("silver.data_items").groupBy("reporting_unit_type_code").count().show()

In [ ]:
# Cell 2 - gold.dim_period (one row per distinct reporting period)
# AIHW mixes standard Australian financial years (Jul-Jun, 12 months) with shorter
# hand-hygiene audit windows - some of which also start in July (Jul-Oct). A true FY
# therefore needs start month 7 AND end month 6; everything else is an audit period
# with an honest date-range label (instead of nonsense like 2013-13).
start = F.to_date("reporting_start_date")
end = F.to_date("reporting_end_date")
is_fy = (F.month(start) == 7) & (F.month(end) == 6)
fy_label = F.concat(F.year(start).cast("string"), F.lit("-"),
                    F.substring(F.year(end).cast("string"), 3, 2))
range_label = F.concat(F.date_format(start, "MMM yyyy"), F.lit(" to "), F.date_format(end, "MMM yyyy"))

periods = (
    spark.table("silver.datasets")
    .select(
        F.when(is_fy, fy_label).otherwise(range_label).alias("period_label"),
        F.when(is_fy, fy_label).alias("financial_year"),
        F.when(is_fy, F.lit("Financial year")).otherwise(F.lit("Audit period")).alias("period_type"),
        F.year(start).alias("period_sort_year"),
        start.alias("period_start_date"),
        end.alias("period_end_date"),
    )
    .distinct()
)

w_period = Window.orderBy("period_start_date", "period_end_date")
dim_period = periods.withColumn("period_key", F.row_number().over(w_period)).select(
    "period_key", "period_label", "financial_year", "period_type",
    "period_sort_year", "period_start_date", "period_end_date"
)

dim_period.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_period")
print(f"gold.dim_period: {dim_period.count()} rows")
dim_period.orderBy("period_key").show(60, truncate=False)

In [ ]:
# Cell 3 - gold.dim_hospital (reporting_unit_type_code = H) + Unknown member (-1)
# State / LHN / PHN attributes are nested in the mapped_reporting_units array;
# pull the first element whose mapping type matches the code we want.
def mapped(col, type_code):
    arr = F.filter(col, lambda x: x["map_type"]["mapped_reporting_unit_code"] == type_code)
    return F.element_at(arr, 1)["mapped_reporting_unit"]

ru = spark.table("silver.reporting_units").where(
    F.col("reporting_unit_type.reporting_unit_type_code") == "H"
)
mc = F.col("mapped_reporting_units")

hospitals = ru.select(
    F.col("reporting_unit_code"),
    F.col("reporting_unit_name"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("private"),
    F.col("closed"),
    mapped(mc, "STATE_MAPPING")["reporting_unit_code"].alias("state_code"),
    mapped(mc, "STATE_MAPPING")["reporting_unit_name"].alias("state_name"),
    mapped(mc, "H_LHN")["reporting_unit_code"].alias("lhn_code"),
    mapped(mc, "H_LHN")["reporting_unit_name"].alias("lhn_name"),
    mapped(mc, "H_PHN")["reporting_unit_code"].alias("phn_code"),
    mapped(mc, "H_PHN")["reporting_unit_name"].alias("phn_name"),
)

w_hosp = Window.orderBy("reporting_unit_code")
hospitals_keyed = hospitals.withColumn("hospital_key", F.row_number().over(w_hosp))

unknown = spark.sql("""
    SELECT -1 AS hospital_key,
           'UNKNOWN' AS reporting_unit_code,
           'Unknown / non-hospital reporting unit' AS reporting_unit_name,
           CAST(NULL AS DOUBLE)  AS latitude,
           CAST(NULL AS DOUBLE)  AS longitude,
           CAST(NULL AS BOOLEAN) AS private,
           CAST(NULL AS BOOLEAN) AS closed,
           CAST(NULL AS STRING)  AS state_code,
           CAST(NULL AS STRING)  AS state_name,
           CAST(NULL AS STRING)  AS lhn_code,
           CAST(NULL AS STRING)  AS lhn_name,
           CAST(NULL AS STRING)  AS phn_code,
           CAST(NULL AS STRING)  AS phn_name
""")

dim_hospital = unknown.unionByName(hospitals_keyed).select(
    "hospital_key", "reporting_unit_code", "reporting_unit_name", "latitude", "longitude",
    "private", "closed", "state_code", "state_name", "lhn_code", "lhn_name", "phn_code", "phn_name"
)

dim_hospital.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_hospital")
print(f"gold.dim_hospital: {dim_hospital.count()} rows (incl. Unknown -1)")
dim_hospital.show(5, truncate=False)

In [ ]:
# Cell 4 - gold.dim_measure (silver.measures_ref + silver.measure_categories)
# A measure carries its categories as an array; take the first (measures are
# single-category in this snapshot) and join measure_categories for the canonical name.
m = spark.table("silver.measures_ref")
cat = spark.table("silver.measure_categories")

measures = m.select(
    F.col("measure_code"),
    F.col("measure_name"),
    F.element_at(F.col("measure_categories"), 1)["measure_category_code"].alias("measure_category_code"),
    F.col("units.units_name").alias("units_name"),
    F.col("units.units_display").alias("units_display"),
    F.col("units.decimal_places").alias("decimal_places"),
    F.col("units.units_are_prefix").alias("units_are_prefix"),
).join(cat, on="measure_category_code", how="left")

w_meas = Window.orderBy("measure_code")
dim_measure = measures.withColumn("measure_key", F.row_number().over(w_meas)).select(
    "measure_key", "measure_code", "measure_name",
    "measure_category_code", "measure_category_name",
    "units_name", "units_display", "decimal_places", "units_are_prefix"
)

dim_measure.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_measure")
print(f"gold.dim_measure: {dim_measure.count()} rows")
dim_measure.orderBy("measure_key").show(40, truncate=False)

In [ ]:
# Cell 5 - gold.dim_reported_measure (distinct reported_measure_code + is_total flag)
# data_items has the codes but not the names; names live in datasets.reported_measure_summary.
# is_total flags genuine aggregate rows ('Total', 'All patients', 'All ...') - NOT procedure
# names that merely contain the word total (e.g. 'Total hip replacement').
rm_names = (
    spark.table("silver.datasets")
    .select(
        F.col("reported_measure_summary.reported_measure_code").alias("reported_measure_code"),
        F.col("reported_measure_summary.reported_measure_name").alias("reported_measure_name"),
    )
    .where(F.col("reported_measure_summary.reported_measure_code").isNotNull())
    .distinct()
)

rm_codes = (
    spark.table("silver.data_items")
    .select("reported_measure_code")
    .where(F.col("reported_measure_code").isNotNull())
    .distinct()
)

lname = F.lower(F.trim(F.col("reported_measure_name")))
reported = rm_codes.join(rm_names, on="reported_measure_code", how="left").withColumn(
    "is_total",
    F.coalesce((lname == "total") | lname.startswith("all ") | (lname == "all"), F.lit(False)),
)

w_rm = Window.orderBy("reported_measure_code")
dim_reported_measure = reported.withColumn("reported_measure_key", F.row_number().over(w_rm)).select(
    "reported_measure_key", "reported_measure_code", "reported_measure_name", "is_total"
)

dim_reported_measure.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_reported_measure")
print(f"gold.dim_reported_measure: {dim_reported_measure.count()} rows")
dim_reported_measure.where("is_total").orderBy("reported_measure_key").show(50, truncate=False)

In [ ]:
# Cell 6 - gold.fact_measure_value (grain: one row per AIHW data item; all reporting-unit types)
start = F.to_date("reporting_start_date")
end = F.to_date("reporting_end_date")
is_fy = (F.month(start) == 7) & (F.month(end) == 6)
fy_label = F.concat(F.year(start).cast("string"), F.lit("-"),
                    F.substring(F.year(end).cast("string"), 3, 2))
range_label = F.concat(F.date_format(start, "MMM yyyy"), F.lit(" to "), F.date_format(end, "MMM yyyy"))
ds_period = spark.table("silver.datasets").select(
    "data_set_id", F.when(is_fy, fy_label).otherwise(range_label).alias("period_label")
)

di = spark.table("silver.data_items")
d_hosp = spark.table("gold.dim_hospital").select("hospital_key", "reporting_unit_code")
d_meas = spark.table("gold.dim_measure").select("measure_key", "measure_code")
d_rm = spark.table("gold.dim_reported_measure").select("reported_measure_key", "reported_measure_code")
d_period = spark.table("gold.dim_period").select("period_key", "period_label")

fact = (
    di
    .join(ds_period, on="data_set_id", how="left")
    .join(d_period, on="period_label", how="left")
    .join(d_meas, on="measure_code", how="left")
    .join(d_rm, on="reported_measure_code", how="left")
    .join(d_hosp, on="reporting_unit_code", how="left")
    .select(
        F.col("measure_key"),
        F.col("reported_measure_key"),
        F.coalesce(F.col("hospital_key"), F.lit(-1)).alias("hospital_key"),
        F.col("period_key"),
        F.col("value"),
        F.col("lower_value"),
        F.col("upper_value"),
        F.col("suppression_count"),
        F.col("caveat_count"),
        F.col("reporting_unit_code"),
        F.col("reporting_unit_name"),
        F.col("reporting_unit_type_code"),
        F.col("data_set_id"),
        F.col("group_number"),
    )
)

fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_measure_value")
print(f"gold.fact_measure_value: {fact.count()} rows")

In [ ]:
# Cell 7 - Gold verification: counts, FK integrity, orphans, is_total double-count test
fact = spark.table("gold.fact_measure_value")

print("=== Row counts ===")
for t in ["dim_period", "dim_hospital", "dim_measure", "dim_reported_measure", "fact_measure_value"]:
    print(f"gold.{t}: {spark.table('gold.' + t).count()}")

print("\n=== Fact FK null check (expect 0 everywhere; hospital uses -1 not null) ===")
fact.select(
    F.sum(F.col("measure_key").isNull().cast("int")).alias("measure_key_null"),
    F.sum(F.col("reported_measure_key").isNull().cast("int")).alias("reported_measure_key_null"),
    F.sum(F.col("period_key").isNull().cast("int")).alias("period_key_null"),
    F.sum(F.col("hospital_key").isNull().cast("int")).alias("hospital_key_null"),
    F.sum((F.col("hospital_key") == -1).cast("int")).alias("hospital_key_unknown"),
).show()

print("=== hospital_key = -1 by reporting_unit_type (should be the non-H types only) ===")
fact.where(F.col("hospital_key") == -1).groupBy("reporting_unit_type_code").count().show()
print("=== hospital_key != -1 by reporting_unit_type (should be H only) ===")
fact.where(F.col("hospital_key") != -1).groupBy("reporting_unit_type_code").count().show()

print("=== dim_period (financial years + audit periods) ===")
spark.table("gold.dim_period").orderBy("period_key").show(60, truncate=False)

print("=== dim_reported_measure is_total flags ===")
spark.table("gold.dim_reported_measure").groupBy("is_total").count().show()
spark.table("gold.dim_reported_measure").where("is_total").show(50, truncate=False)

print("=== Variant double-count check: measure/hospital/period grains with >1 reported-measure variant ===")
(
    fact.where((F.col("hospital_key") != -1) & F.col("value").isNotNull())
    .groupBy("measure_key", "hospital_key", "period_key")
    .agg(F.countDistinct("reported_measure_key").alias("n_variants"))
    .where(F.col("n_variants") > 1)
    .orderBy(F.col("n_variants").desc())
    .show(5)
)